In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

class BacktestEngine:
    def __init__(self, data, strategy, initial_balance=10000, benchmark_returns=None, risk_free_rate=0.02):
        """
        data: Pandas DataFrame with 'open', 'high', 'low', 'close' columns
        strategy: Function that generates trading signals
        initial_balance: Starting cash balance
        benchmark_returns: Pandas Series of daily returns of the benchmark (e.g., S&P 500)
        risk_free_rate: Daily risk-free rate (default is 2% annualized)
        """
        self.data = data.copy()
        self.strategy = strategy
        self.initial_balance = initial_balance
        self.balance = initial_balance
        self.position = 0  # Number of shares held
        self.trades = []  # Stores trade details
        self.data['signal'] = 0  # Placeholder for trading signals
        self.benchmark_returns = benchmark_returns
        self.risk_free_rate = risk_free_rate / 252  # Convert annual risk-free rate to daily
        self.run_backtest()

    def place_order(self, order_type, price, size):
        """
        Executes a trade and updates balance and position.
        order_type: 'buy' or 'sell'
        price: execution price
        size: number of shares
        """
        cost = price * size
        if order_type == 'buy' and self.balance >= cost:
            self.position += size
            self.balance -= cost
            self.trades.append({'type': 'buy', 'price': price, 'size': size})
        elif order_type == 'sell' and self.position >= size:
            self.position -= size
            self.balance += cost
            self.trades.append({'type': 'sell', 'price': price, 'size': size})
        elif order_type == 'short' and self.balance >= cost:
            self.position -= size
            self.balance -= cost
            self.trades.append({'type': 'short', 'price': price, 'size': size})
        elif order_type == 'close' and self.position >= size:
            self.balance += self.position * price
            self.position = 0
            self.trades.append({'type': 'close', 'price': price, 'size': size})

    def run_backtest(self):
        """ Executes the strategy over historical data """
        self.daily_returns = []  # Store strategy daily returns
        self.benchmark_returns_strategy = []  # Store benchmark returns for calculating alpha
        for i in range(len(self.data)):
            row = self.data.iloc[i]
            signal = self.strategy(row)
            self.data.at[row.name, 'signal'] = signal
            
            if signal == 1 and self.position == 0:  # Buy signal
                self.place_order('buy', row['close'], size=10)
            elif signal == -1 and self.position > 0:  # Sell signal
                self.place_order('sell', row['close'], size=10)
            
            # Calculate strategy daily return and benchmark return
            if i > 0:  # Ignore the first row for returns
                prev_row = self.data.iloc[i-1]
                strategy_return = (self.balance + self.position * row['close']) / (self.balance + self.position * prev_row['close']) - 1
                self.daily_returns.append(strategy_return)
                self.benchmark_returns_strategy.append(self.benchmark_returns[i] if self.benchmark_returns is not None else 0)
        
    def plot_results(self):
        """ Plots price, signals, and trades """
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=self.data.index, y=self.data['close'], mode='lines', name='Price'))

        # Buy signals
        buy_signals = self.data[self.data['signal'] == 1]
        fig.add_trace(go.Scatter(x=buy_signals.index, y=buy_signals['close'], mode='markers',
                                 marker=dict(color='green', symbol='triangle-up', size=10), name='Buy'))

        # Sell signals
        sell_signals = self.data[self.data['signal'] == -1]
        fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['close'], mode='markers',
                                 marker=dict(color='red', symbol='triangle-down', size=10), name='Sell'))

        fig.update_layout(title='Backtest Results', xaxis_title='Time', yaxis_title='Price')
        fig.show()

    def get_performance(self):
        """ Calculates key performance metrics """
        final_balance = self.balance + (self.position * self.data['close'].iloc[-1])
        strategy_returns = np.array(self.daily_returns)
        benchmark_returns = np.array(self.benchmark_returns_strategy)

        # Sharpe Ratio Calculation
        excess_returns = strategy_returns - self.risk_free_rate
        sharpe_ratio = excess_returns.mean() / excess_returns.std() if excess_returns.std() != 0 else 0
        
        # Alpha Calculation (Using Benchmark)
        if len(benchmark_returns) > 0:
            covariance = np.cov(strategy_returns, benchmark_returns)[0][1]
            benchmark_variance = np.var(benchmark_returns)
            beta = covariance / benchmark_variance if benchmark_variance != 0 else 0
            alpha = strategy_returns.mean() - (self.risk_free_rate + beta * (benchmark_returns.mean() - self.risk_free_rate))
        else:
            alpha = 0  # No benchmark data

        return {
            'Initial Balance': self.initial_balance,
            'Final Balance': round(final_balance, 2),
            'Net Profit': round(final_balance - self.initial_balance, 2),
            'Total Trades': len(self.trades),
            'Sharpe Ratio': round(sharpe_ratio, 2),
            'Alpha': round(alpha, 2)
        }